# Lab 2: Processing Millions of Contribution Records

**Time:** about 60 minutes

In Lab 1 the challenge was the model. Now the model doesn't matter: the challenge is **size**.

Every month, every employer posts a contribution line for every member. Over nine years that's about 2.4 million rows
for this synthetic fund. The real Fund's history is many times larger. Your job is to turn those rows into
**AI-ready feature tables**: one row per employer and one row per member, holding the numbers a model or an analyst
would want (average contribution, how often they pay late, how their headcount is changing, and so on).

You'll do it twice:

- **Part A:** with pandas, on one machine, to see where it starts to strain;
- **Part B:** with **Dask**, a library that splits the data into pieces and processes them in parallel, and which can
  run the same code on a laptop or on a cluster of many machines.

```
                 member_contributions/ (Parquet, one folder per year)
                                  │
                         split into partitions
                 ┌────────┬───────┼───────┬────────┐
                 P1       P2      P3      P4  ...  Pn
                 │        │       │       │        │
             filter · clean · compute (in parallel on each partition)
                 └────────┴───────┼───────┴────────┘
                           combine partial results
                                  │
                 employer_features.parquet, member_history.parquet
```

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **memory (RAM)** | the computer's working space; a pandas table must fit in it completely |
| **Parquet** | a compressed file format that stores data column by column; smaller and faster than CSV, and it remembers column types |
| **partition** | one piece of a big table; Dask works on several pieces at the same time |
| **parallel** | doing several pieces of work at once, one per CPU core |
| **lazy evaluation** | writing down *what* to compute without computing it yet; the work only happens when you ask for a result |
| **aggregate** | to summarise many rows into fewer (e.g. sum per employer per month) |
| **feature table** | a table with one row per thing (employer, member) and one column per useful number about it |
| **reconcile** | to check your totals against an independent source, to prove they're right |

## Where this lab fits

**What you've done so far.** On Day 1 you built a pipeline for the employer-level tables, which are small enough to
load in one go. On Day 2 and in Lab 1 today you built models on about 7,800 claims. Every table so far has fitted in
your laptop's memory with room to spare.

**What you'll do now.** The member-level contribution history is different: about 2.4 million rows here, and many times
that at the real Fund. You'll turn it into two feature tables (one row per employer, one row per member), first with
pandas to see where it strains, then with Dask, which processes the data in pieces. Then you'll prove your totals are
right by checking them against Day 1's employer remittances.

**Why it matters.** The most useful signals about members and employers are in the detailed history: who stopped
posting, whose headcount is falling, who pays late. Models can only use those signals if someone can compute them
reliably, at full size, every month.

**What happens next.** Lab 3 takes this same feature job and turns it into a versioned pipeline with DVC, so that
anyone can rebuild exactly these files later, and see what changes when a rule changes.

### Theory: what to do when data doesn't fit

pandas works **in memory**: the whole table has to be loaded into RAM, and operations like `groupby` and `merge` often
need two or three times that while they work. When the data outgrows memory, you have three options:

| Option | Idea | Example |
|---|---|---|
| **Make the data smaller** | better types, fewer columns, fewer rows | `category` instead of text; read only the columns you need |
| **Scale up** | a bigger machine | a server with 256 GB of RAM. Simple, but there's always a limit, and it's expensive |
| **Scale out** | split the data into pieces and process them separately, possibly on many machines | Dask, Spark |

Scaling out works because most feature jobs can be done **piece by piece**. To get total contributions per employer,
you can sum each piece separately and then add up the partial sums. No single step ever needs all 2.4 million rows at
once. That idea, splitting a job into pieces that run separately and then combining the results, is the basis of every
"big data" tool.

## Imports and settings

`dask` is the new library. `dask.dataframe` (imported as `dd`) gives tables that look and behave almost exactly like
pandas DataFrames, but can be far bigger than memory.

Two folder paths are set here: `SRC` is where the raw data lives, and `OUT` is where the finished feature tables will be
written (created if it doesn't exist).

In [ ]:
import os
import time                        # to time each step
from pathlib import Path

import numpy as np
import pandas as pd
import dask                        # parallel computing
import dask.dataframe as dd        # Dask's pandas-like tables

# Show more columns when printing tables.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# Where the raw data is, and where the feature tables will go.
SRC = Path("member_contributions")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)       # parents=True also creates "data" if needed

# os.cpu_count() = how many CPU cores this machine has. Dask can use all of them.
print("dask", dask.__version__, "| CPU cores:", os.cpu_count())

Look at how the data is stored on disk. Instead of one huge file, there's one folder per year, each holding a
Parquet file.

In [ ]:
# rglob("*.parquet") finds every .parquet file in the folder and its sub-folders.
for path in sorted(SRC.rglob("*.parquet")):
    print(path.as_posix())

> **Where we are.** The libraries are loaded, and you've seen how the data is stored: Parquet files, one folder per
> year. Part A tries the job with plain pandas, so you can see how far a single table in memory gets you.

This is how large datasets are usually stored:

- **Parquet** files rather than CSV. A CSV file stores everything as text, row after row. Parquet stores each column
  together and compresses it, so files are much smaller, reading only some columns is fast, and the column types
  (numbers, dates, text) are kept.
- **One folder per year**, named `year=2023`, `year=2024` and so on. This naming convention is called *Hive
  partitioning*, and both pandas and Dask understand it: the folder name becomes a column called `year`, and a question
  that only needs 2023 can skip the other folders entirely.

## Part A: the single-machine baseline (pandas)

Start small: read one year with pandas, and time it. `memory_usage(deep=True)` reports how much RAM the table takes,
counting the text columns properly.

In [ ]:
start = time.perf_counter()
y2025 = pd.read_parquet(SRC / "year=2025")          # just the 2025 folder
seconds = time.perf_counter() - start

memory_mb = y2025.memory_usage(deep=True).sum() / 1e6     # bytes -> megabytes
print(f"2025: {len(y2025):,} rows in {seconds:.2f}s, {memory_mb:.0f} MB in memory")
y2025.head()

Each row is one contribution posted to one member's account:

| Column | Meaning |
|---|---|
| `member_no`, `employer_no` | Who, and through which employer |
| `period` | The wage month (`YYYY-MM`) |
| `gross_wage_tzs` | The member's gross wage that month |
| `employee_share_tzs`, `employer_share_tzs` | 10% each |
| `posted_date` | When the contribution was posted to the member's account |
| `entry_type` | `REGULAR`, or `ARREARS` if it was posted more than 60 days after the due date |

Now a typical feature job: **monthly totals per employer**. For each employer and each month, count the members and
add up the contributions. In pandas that's a `groupby` (group the rows by employer and month) followed by `agg`
(say how to summarise each group).

In [ ]:
start = time.perf_counter()

# Total contribution = employee share + employer share.
y2025["total"] = y2025["employee_share_tzs"] + y2025["employer_share_tzs"]

# One row per employer per month: how many different members, and how much in total.
emp_month_2025 = y2025.groupby(["employer_no", "period"]).agg(
    members=("member_no", "nunique"),       # nunique = number of different values
    contributions=("total", "sum"),
)
print(f"{len(emp_month_2025):,} employer-months in {time.perf_counter() - start:.2f}s")

Fine for one year. Now load all nine years at once and watch the memory figure:

In [ ]:
start = time.perf_counter()
all_years = pd.read_parquet(SRC)                    # every year folder at once
seconds = time.perf_counter() - start

memory_mb = all_years.memory_usage(deep=True).sum() / 1e6
print(f"all years: {len(all_years):,} rows in {seconds:.1f}s, {memory_mb:,.0f} MB in memory")

It works on a laptop, but look at the memory figure and do the arithmetic for the real Fund: several times more
members, more years, and more columns in a real extract. A table that takes 1 GB here can easily become 20-50 GB.

**pandas needs the whole table in memory at once**, and usually two or three times that while it works, because
operations like `groupby` and `merge` make temporary copies. That's the point where a laptop, and often a single server,
runs out of memory. There are two ways round it: make the data smaller (next exercise), or process it in pieces
(Part B).

**Your turn (1).** Before moving on, find out how much memory you could save in pandas alone. Convert
`member_no`, `employer_no`, `period` and `entry_type` to `category`, and the three money columns to
`int32` or `int64` as appropriate. How far does the memory figure drop? Would `int32` be safe for
`gross_wage_tzs`? (Hint: `np.iinfo(np.int32).max`.)

In [ ]:
# Your code here

`del` removes a variable, which lets Python give the memory back. Do it before Part B so you don't run out.

In [ ]:
# Free the memory before Part B.
del all_years

> **Where we are.** pandas handled one year easily and all nine years at a noticeable memory cost. You've seen that
> better types can shrink a table a lot, but at the real Fund's size that's still not enough. Part B processes the
> data in pieces instead.

## Part B: the same job with Dask

A **Dask DataFrame** looks and behaves like a pandas DataFrame, but under the hood it's a collection of many smaller
pandas DataFrames (**partitions**) plus a **plan** of what to do with them.

### What a "partition" is, and why it helps

Imagine a 2.4-million-row table cut into nine pieces, one per year. Dask doesn't load all nine pieces. It keeps a list of
the pieces and a plan of the steps to apply. When you finally ask for a result, Dask:

1. loads a few pieces at a time (as many as you have CPU cores);
2. applies the steps to each piece **at the same time**, one piece per core;
3. combines the small partial results into the final answer;
4. forgets each piece once it's done with it.

Because only a few pieces are in memory at any moment, the whole dataset never has to fit at once. And because several
cores work at the same time, a large job finishes faster.

In [ ]:
# This doesn't read the data yet. It only looks at the files and makes a plan.
ddf = dd.read_parquet(SRC)
print("partitions:", ddf.npartitions)
ddf

That output is **not data**. It's a description of the table: the column names, their types, and how many partitions
there are. Making it took almost no time, because nothing has been read yet. This is called **lazy evaluation**: Dask
waits until you actually need a result.

`head()` is an exception: it reads just enough of the first partition to show you a few real rows.

### Theory: lazy evaluation and the task plan

In pandas, every line runs immediately: `valid = df[df["wage"] > 0]` filters the data there and then. In Dask, the same
line only **writes down** the step. Nothing is read or computed until you call `.compute()` (or `dask.compute(...)`).

Why wait? Because once Dask has the whole plan, it can organise the work cleverly:

- It reads each file **once**, even if the plan uses it for several results.
- It skips columns and folders the plan never uses.
- It runs independent pieces at the same time on different CPU cores.
- It throws away each piece's intermediate results as soon as they're no longer needed, so memory stays low.

Behind the scenes, the plan is a **task graph**: a diagram of small tasks ("read partition 3", "filter partition 3",
"sum partition 3", "add the partial sums") and which ones depend on which. You never have to draw it, but it explains
the habits in this lab: build the whole plan first, then compute once.

In [ ]:
ddf.head()

### Step 1: filter out invalid records

These rules come from the same thinking as Day 1's validation stage: decide what a valid row looks like, and keep only
those. Four rules:

- the wage must be **more than zero**;
- the wage must be **at most TZS 50 million** a month (anything higher is treated as a typing error);
- the **employer number** must be filled in;
- the **posted date** must be filled in.

The code is exactly what you'd write in pandas. Each rule is a True/False column; `&` means "and", so a row is kept only
if all four are True. Notice that every line returns instantly: you're only adding steps to the plan.

In [ ]:
MAX_WAGE = 50_000_000     # TZS per month; above this is treated as a data-entry error

# Each rule is a True/False column.
wage_positive = ddf["gross_wage_tzs"] > 0
wage_not_too_big = ddf["gross_wage_tzs"] <= MAX_WAGE
has_employer = ddf["employer_no"].notnull()
has_posted_date = ddf["posted_date"].notnull()

# Keep the rows that pass all four rules.
valid = ddf[wage_positive & wage_not_too_big & has_employer & has_posted_date]

# Remove rows that are identical in every column, across all partitions.
valid = valid.drop_duplicates()

Add two new columns to the plan: the **total contribution** (employee + employer share), and a **1/0 flag** for postings
that arrived as arrears (more than 60 days late). The average of a 1/0 column is the share of 1s, which will give each
employer's "share of postings that were late".

In [ ]:
# Two new columns: the total contribution, and a 1/0 flag for arrears postings.
valid["total_tzs"] = valid["employee_share_tzs"] + valid["employer_share_tzs"]
valid["is_arrears"] = (valid["entry_type"] == "ARREARS").astype("int8")    # int8 = a very small whole number

print(f"Still nothing computed. `valid` is a plan that will run over {valid.npartitions} partitions.")

How many rows did the rules remove? Now you're asking for actual numbers, so Dask does the work, in parallel across your
CPU cores.

`dask.compute(a, b)` runs both plans together and returns real numbers. `shape[0]` is the number of rows.

In [ ]:
start = time.perf_counter()

# shape[0] (the number of rows) is part of the plan too. dask.compute() runs it and gives back real numbers.
n_raw, n_valid = dask.compute(ddf.shape[0], valid.shape[0])

print(f"{n_raw:,} raw rows -> {n_valid:,} valid rows ({n_raw - n_valid:,} removed) in {time.perf_counter() - start:.1f}s")

Why compute both numbers in one call? If you'd asked for `ddf.shape[0].compute()` and then `valid.shape[0].compute()`
separately, Dask would have read all the files from disk twice. One `dask.compute` call reads them once and works out
both answers on the way. With big data, reading from disk is usually the slowest part, so this matters.

**Your turn (2).** How many rows fail each rule separately? Compute all four counts (wage ≤ 0, wage above
`MAX_WAGE`, missing employer, missing posted date) in a single `dask.compute` call.

In [ ]:
# Your code here

### Theory: how a group-by works when the data is in pieces

When Dask sums contributions per employer per month, it can't see all of an employer's rows at once: they're spread
across partitions. So it does the job in two rounds:

```
partition 1:  E001/2024-03 → partial sum 4.1m      ┐
partition 2:  E001/2024-03 → partial sum 0.2m      ├─► combine: E001/2024-03 = 4.3m
partition 3:  (no E001 rows for that month)        ┘
```

1. On each partition, compute a **partial result** for every group.
2. Combine the partial results across partitions.

This works directly for **sums, counts, minimums and maximums**. A **mean** works too, because Dask keeps the partial
sums and partial counts separately and divides only at the end (averaging the partial averages would give the wrong
answer if the partitions have different numbers of rows).

Some calculations can't be split this way: a median, a sort, or anything that needs one employer's months in order.
Those are why the next steps reduce the data with Dask first, then switch to pandas.

### Step 2: employer-month totals

This is where the size drops dramatically: 2.4 million member rows become about 120,000 employer-month rows (one per
employer per month).

The plan groups by employer and month, and says how to combine each column:

- `member_no: count`: how many member postings there were;
- `gross_wage_tzs: sum` and `total_tzs: sum`: total wages and total contributions;
- `is_arrears: mean`: the share of postings that were arrears (the average of a 1/0 column).

`.compute()` at the end runs the plan.

In [ ]:
start = time.perf_counter()

# The plan: group by employer and month, and say how to combine each column.
emp_month_plan = valid.groupby(["employer_no", "period"]).agg({
    "member_no": "count",          # number of member postings
    "gross_wage_tzs": "sum",
    "total_tzs": "sum",
    "is_arrears": "mean",          # share of postings that were arrears
})

# .compute() runs the plan and returns an ordinary pandas DataFrame.
emp_month = emp_month_plan.compute()
print(f"{len(emp_month):,} employer-months in {time.perf_counter() - start:.1f}s")

Give the columns clearer names, and use `reset_index()` to turn `employer_no` and `period` (which the groupby moved into
the row labels) back into ordinary columns. `type(...)` confirms the result is now a plain pandas table.

In [ ]:
emp_month = emp_month.rename(columns={
    "member_no": "members_posted",
    "gross_wage_tzs": "wages_tzs",
    "total_tzs": "contributions_tzs",
    "is_arrears": "arrears_share",
})
emp_month = emp_month.reset_index()
print(type(emp_month))
emp_month.head()

That `.compute()` turned a big Dask plan into a small pandas table that fits comfortably in memory. This is the most
important pattern in large-scale data processing:

> **Use the distributed engine (Dask, Spark) to reduce big data to small data. Then use pandas on the small data.**

Some calculations need each employer's months **in time order**: growth from last month, a rolling 3-month average.
They're awkward to do across partitions, but easy on the small table. So do them in pandas:

- `pct_change()`: the change from the previous month, as a share (0.05 = up 5%);
- `pct_change(3)`: the change from three months earlier;
- `rolling(3).mean()`: the average of this month and the two before it, which smooths out one-off spikes.

Grouping by employer first makes sure one employer's months are never mixed with another's.

### Theory: features that need time order

Some of the most useful features compare a month with the months before it. They're called **window** or **lag**
features:

| Feature | Calculation | Example for one employer |
|---|---|---|
| change from last month | `pct_change()` | 4.0m → 4.4m = +0.10 (up 10%) |
| change from 3 months ago | `pct_change(3)` | Jan 4.0m → Apr 3.6m = -0.10 |
| 3-month rolling average | `rolling(3).mean()` | (4.0 + 4.4 + 4.1) / 3 = 4.17m |

Two things must be right for these to mean anything:

1. **The rows must be sorted by month** within each employer. Otherwise "the previous row" isn't last month.
2. **The calculation must restart for each employer.** Without `groupby("employer_no")`, the first month of employer B
   would be compared with the last month of employer A.

Rolling averages smooth out one-off spikes, such as a bonus month, so the underlying trend is easier to see. You'll
use these same tools for forecasting on Day 4.

In [ ]:
# Put each employer's months in order.
emp_month = emp_month.sort_values(["employer_no", "period"])

# Average contribution per member.
emp_month["contribution_per_member_tzs"] = (emp_month["contributions_tzs"] / emp_month["members_posted"]).round()

# Group by employer, so the next calculations never mix two employers.
by_employer = emp_month.groupby("employer_no")["contributions_tzs"]

# Change from the previous month (0.05 = +5%) and from three months earlier.
emp_month["growth_1m"] = by_employer.pct_change(fill_method=None)
emp_month["growth_3m"] = by_employer.pct_change(3, fill_method=None)

# Average of this month and the two before it. reset_index(level=0, drop=True) lines the result up with the table again.
emp_month["rolling_3m_avg_tzs"] = by_employer.rolling(3, min_periods=1).mean().reset_index(level=0, drop=True).round()

emp_month.tail()

> **Where we are.** 2.4 million member rows have become about 120,000 employer-month rows, small enough for pandas,
> and you've added month-on-month growth and rolling averages. A model needs one row per employer, so the next step
> summarises each employer's months into a single row.

### Step 3: one row per employer, the AI-ready feature table

A model that predicts "will this employer fall behind?" needs **one row per employer**, with each column describing
their recent behaviour. Using the months from January 2024 onwards, summarise each employer:

| Feature | What it tells you |
|---|---|
| `months_with_postings` | how many months they posted anything |
| `first_period`, `last_period` | when they started and last posted |
| `avg_members` | typical headcount |
| `avg_contribution_tzs`, `avg_contribution_per_member_tzs` | how much they typically pay |
| `arrears_share` | how often postings arrived late |
| `growth_3m_latest` | the most recent 3-month growth |
| `contribution_volatility` | how much the monthly amount jumps around (the standard deviation of monthly growth) |
| `months_missing_since_2024` | months with no postings at all |

In [ ]:
# Only months from January 2024.
recent = emp_month[emp_month["period"] >= "2024-01"]

# One row per employer. Each line says: new column = (existing column, how to combine).
employer_features = recent.groupby("employer_no").agg(
    months_with_postings=("period", "nunique"),
    first_period=("period", "min"),
    last_period=("period", "max"),
    avg_members=("members_posted", "mean"),
    avg_contribution_tzs=("contributions_tzs", "mean"),
    avg_contribution_per_member_tzs=("contribution_per_member_tzs", "mean"),
    arrears_share=("arrears_share", "mean"),
    growth_3m_latest=("growth_3m", "last"),
    contribution_volatility=("growth_1m", "std"),     # how much the monthly growth jumps around
)
employer_features = employer_features.round(4).reset_index()

# 24 months from January 2024 to December 2025. Months without postings are gaps.
employer_features["months_missing_since_2024"] = 24 - employer_features["months_with_postings"]

print(employer_features.shape)
# describe().T: one row per column, with count, average, spread, min and max.
employer_features.describe().T.round(2)

Save it as Parquet, so the next person (or the model in Lab 3) can read it quickly with the column types intact.

In [ ]:
# Save it.
employer_features.to_parquet(OUT / "employer_features.parquet", index=False)
print("saved", OUT / "employer_features.parquet")

> **Where we are.** The employer feature table is saved. The same raw data can be summarised in a different way, one
> row per member, for models and analysis about members rather than employers.

### Step 4: one row per member

The member history is a different summary of the same data, with about 60,000 rows (one per member). Dask is still worth
using here, because the grouping has to go through all 2.4 million rows.

For each member you want: how many months they've had contributions posted, their first and last month, their average
and highest wage, their share of late postings, and how many different employers they've worked for.

**A practical speed tip first.** Finding each member's first and last month means taking the minimum and maximum of
`period`. But `period` is text (`"2024-07"`), and comparing text across 60,000 groups is slow: every comparison has to
look at the characters one by one. Turn the month into a **number** instead (`year × 12 + month`, so July 2024 becomes
24,295 and later months always get bigger numbers), and the same job runs many times faster.

If a large job is unexpectedly slow, check whether you're summarising text that could be numbers.

In [ ]:
start = time.perf_counter()

# "2024-07" -> year 2024, month 7 -> 2024 * 12 + 7 = 24295. Later months always get bigger numbers.
year = valid["period"].str[:4].astype("int32")       # the first 4 characters, as a number
month = valid["period"].str[5:7].astype("int32")     # characters 6-7, as a number
valid["month_index"] = year * 12 + month

# The plan for the per-member numbers: several summaries per column.
member_stats_plan = valid.groupby("member_no").agg({
    "month_index": ["count", "min", "max"],          # how many months, first, last
    "gross_wage_tzs": ["mean", "max"],               # average and highest wage
    "is_arrears": "mean",                            # share of late postings
})

# Distinct counts (how many different employers) need their own step in Dask.
member_employers_plan = valid.groupby("member_no")["employer_no"].nunique()

# Run both plans in one pass over the data.
member_stats, member_employers = dask.compute(member_stats_plan, member_employers_plan)
print(f"computed in {time.perf_counter() - start:.1f}s")

The result has two-level column names like `("month_index", "min")`, because one column was summarised several ways.
Replace them with simple names. Then turn the month numbers back into readable `YYYY-MM` text with the reverse
calculation:

- year = (number - 1) ÷ 12, rounded down (`//` is whole-number division);
- month = the remainder of (number - 1) ÷ 12, plus 1 (`%` gives the remainder);
- `zfill(2)` pads the month to two digits, so 7 becomes "07".

In [ ]:
# The aggregation gave two-level column names like ("month_index", "min"). Replace them with simple names.
member_stats.columns = ["months_posted", "first_month", "last_month", "avg_wage_tzs", "max_wage_tzs", "arrears_share"]

# Turn the month numbers back into "YYYY-MM" text.
first_year = (member_stats["first_month"] - 1) // 12
first_mon = (member_stats["first_month"] - 1) % 12 + 1
member_stats["first_period"] = first_year.astype(str) + "-" + first_mon.astype(str).str.zfill(2)

last_year = (member_stats["last_month"] - 1) // 12
last_mon = (member_stats["last_month"] - 1) % 12 + 1
member_stats["last_period"] = last_year.astype(str) + "-" + last_mon.astype(str).str.zfill(2)

# Check the conversion on a few rows.
member_stats[["first_month", "first_period", "last_month", "last_period"]].head()

Finally, drop the helper number columns, add the number of employers (pandas lines it up by member number), and save.

In [ ]:
# Drop the helper columns, add the number of employers, and save.
member_history = member_stats.drop(columns=["first_month", "last_month"])
member_history["employers"] = member_employers
member_history = member_history.reset_index()       # member_no back as an ordinary column

member_history.to_parquet(OUT / "member_history.parquet", index=False)
print(f"{len(member_history):,} members")
member_history.head()

> **Where we are.** You've produced both feature tables. They were computed quickly and they look sensible. But
> "looks sensible" isn't proof.

### Theory: reconciliation

**Reconciliation** means checking a number you computed against the same number from an **independent source**.
Accountants do it every month: the bank statement must agree with the cash book. If the two disagree, at least one is
wrong, and the size of the gap tells you how much to worry.

For a feature pipeline, the independent source is the employer remittance table from Day 1, which was recorded by a
different process (the employer's payment) from the member postings. If the member postings for a month add up to
what the employer actually paid, nothing was lost or doubled along the way.

Two habits make reconciliation useful:

- **Compare like with like.** Only months paid in full are compared, because a partial payment won't match the full
  member total.
- **Look at the worst cases, not just the average.** "99.8% reconciled" sounds fine until the 0.2% turns out to be
  your largest employer.

## Does it add up? Reconcile against the employer remittances

A feature pipeline that's fast but wrong is worse than useless, because people will trust its numbers. So check them
against an **independent source**: the employer-level remittances from Day 1 (what each employer actually paid each
month).

The logic: for any month an employer **paid in full**, the member-level postings should add up to exactly what the
employer paid. If they don't, something was lost or doubled along the way.

The steps:

1. keep the months where the employer's payment equals the amount due (paid in full);
2. put each payment next to your member-level total for the same employer and month (`merge`);
3. measure how far apart they are, as a share of the payment.

In [ ]:
remit = pd.read_csv("contributions_monthly_clean.csv", dtype={"employer_no": "string"})

# Months the employer paid in full (paid = due, to the shilling).
paid_in_full = remit[remit["amount_paid_tzs"].round() == remit["contribution_due_tzs"].round()]

# Put the employer's payment next to our member-level total for the same employer and month.
check = paid_in_full.merge(emp_month, on=["employer_no", "period"], how="inner")

# How far apart are they, as a share of the payment? 0 = identical; 0.01 = 1% apart.
check["diff_share"] = (check["contributions_tzs"] - check["amount_paid_tzs"]).abs() / check["amount_paid_tzs"]

print(f"{len(check):,} fully paid employer-months compared")
print(f"within 0.1%: {(check['diff_share'] < 0.001).mean():.1%}")

Now look at the worst mismatches: the employer-months where the two numbers are furthest apart.

In [ ]:
# The five biggest mismatches.
worst5 = check.sort_values("diff_share", ascending=False).head(5)
worst5[["employer_no", "period", "amount_paid_tzs", "contributions_tzs", "diff_share"]]

Nearly every month reconciles to the shilling. The few that don't are worth a look. Most will be months where
you removed a posting in Step 1 (for example a duplicate, or an implausible wage), so the member total is
now lower than the employer's payment. That's exactly the kind of difference a finance team would want
explained before trusting the features.

**Your turn (3).** Pick the worst mismatch from the table above and find the member-level rows for that
employer and period in the **raw** data (`ddf`). Which rule removed something, and was that the right
decision?

In [ ]:
# Your code here

> **Where we are.** The features reconcile with the employer payments, and you've traced the few mismatches to rows
> your validity rules removed. The pipeline is right. The remaining question is when this extra machinery is worth
> using at all.

## Pandas vs Dask: when is it worth it?

Time the same small job both ways: the total employee share per employer, reading only the two columns needed
(`columns=[...]`, which Parquet makes fast).

In [ ]:
# pandas: read the two columns into memory, then group.
start = time.perf_counter()
table = pd.read_parquet(SRC, columns=["employer_no", "employee_share_tzs"])
totals_pandas = table.groupby("employer_no")["employee_share_tzs"].sum()
pandas_seconds = time.perf_counter() - start

# Dask: make a plan for the same thing, then compute it.
start = time.perf_counter()
plan = dd.read_parquet(SRC, columns=["employer_no", "employee_share_tzs"])
totals_dask = plan.groupby("employer_no")["employee_share_tzs"].sum().compute()
dask_seconds = time.perf_counter() - start

print(f"pandas {pandas_seconds:.2f}s | dask {dask_seconds:.2f}s")

On a dataset this size the two are close, and pandas can even win. Dask has overhead: planning the work,
scheduling tasks, combining results. It pays off when:

- the data **doesn't fit in memory** (Dask processes it partition by partition, so memory stays low),
- there's a lot of computation per row and several cores to share it, or
- you move the same code to a **cluster** with `dask.distributed`, where more machines mean more memory and
  more cores.

If your data fits comfortably in memory, pandas is simpler. Don't add distributed processing because it
sounds serious.

## And Spark?

Apache Spark does the same job at a larger scale. It's the usual choice when a data platform already runs
it (Databricks, a Hadoop cluster, or a cloud service such as Azure Synapse, AWS EMR or Google Dataproc).
The code has the same shape. Here's Step 1 and Step 2 in PySpark, for reference. You don't need to run it;
it requires Java and a Spark installation.

```python
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("contributions").getOrCreate()
df = spark.read.parquet("member_contributions")          # reads year=... folders as a column too

valid = df.filter((F.col("gross_wage_tzs") > 0) & (F.col("gross_wage_tzs") <= 50_000_000)
                  & F.col("employer_no").isNotNull() & F.col("posted_date").isNotNull())
valid = valid.dropDuplicates()
valid = valid.withColumn("total_tzs", F.col("employee_share_tzs") + F.col("employer_share_tzs"))
valid = valid.withColumn("is_arrears", (F.col("entry_type") == "ARREARS").cast("int"))

emp_month = valid.groupBy("employer_no", "period").agg(
    F.count("member_no").alias("members_posted"),
    F.sum("gross_wage_tzs").alias("wages_tzs"),
    F.sum("total_tzs").alias("contributions_tzs"),
    F.avg("is_arrears").alias("arrears_share"),
)
emp_month.write.mode("overwrite").parquet("data/processed/employer_month")
```

| | pandas | Dask | Spark |
|---|---|---|---|
| Runs on | One machine, in memory | Laptop to cluster | Cluster (also runs locally for testing) |
| Language | Python | Python, pandas-like API | Python (PySpark), SQL, Scala, Java |
| Setup | None | `pip install dask` | Java plus Spark, or a managed platform |
| Sweet spot | Up to a few GB | Tens to hundreds of GB, Python teams | Hundreds of GB and beyond, shared data platforms |

## Wrap-up

You now have two AI-ready tables in `data/processed/`: `employer_features.parquet` and
`member_history.parquet`. In Lab 3 you'll build this same job as a versioned pipeline, so anyone can reproduce
exactly these files, and you'll see what happens when someone changes a rule.

## Looking back, looking ahead

**What you did in this lab.** You took 2.4 million member-level postings and:

1. measured how much memory pandas needs, and how much better types save;
2. built the same job in Dask: a lazy plan, validity rules, and one `compute` that reads the data once;
3. reduced the data to employer-months, then switched to pandas for time-ordered features;
4. produced one-row-per-employer and one-row-per-member feature tables;
5. reconciled your totals against an independent source and traced the mismatches;
6. compared pandas, Dask and Spark, and when each is worth using.

**The ideas to carry forward.**

- Use the big-data engine to make big data small, then use pandas on the small data.
- Build the whole plan first, then compute once.
- Fast but unreconciled numbers aren't ready to use.
- Don't add distributed processing unless the data needs it.

---

## Answers to the "Your turn" exercises

**(1)** Shrinking pandas memory. Text columns with only a few thousand different values (member numbers, employer
numbers, months, entry types) are much smaller as `category` (stored once, then referred to by a small number). Money
columns can use a smaller whole-number type if the values fit.

In [ ]:
small = pd.read_parquet(SRC)
before_mb = small.memory_usage(deep=True).sum() / 1e6

# Text columns with many repeats -> category.
small["member_no"] = small["member_no"].astype("category")
small["employer_no"] = small["employer_no"].astype("category")
small["period"] = small["period"].astype("category")
small["entry_type"] = small["entry_type"].astype("category")

# The two shares fit comfortably in int32.
small["employee_share_tzs"] = small["employee_share_tzs"].astype("int32")
small["employer_share_tzs"] = small["employer_share_tzs"].astype("int32")

after_mb = small.memory_usage(deep=True).sum() / 1e6
print(f"{before_mb:,.0f} MB -> {after_mb:,.0f} MB")

In [ ]:
# Is int32 big enough for gross_wage_tzs? Compare its largest possible value with the largest wage in the data.
print("largest int32 value:", np.iinfo(np.int32).max)
print("largest gross wage: ", small["gross_wage_tzs"].max())
del small

`int32` tops out at about 2.1 billion. Each share is 10% of a wage, so int32 is safe for the shares. The raw
`gross_wage_tzs` column contains a few typing errors that are 100 times too large, so check before
downcasting it. If a value doesn't fit, it wraps around to a wrong number **without any error**.

**(2)** Rule-by-rule counts in one pass. Plan four counts (each `.sum()` of a True/False column counts the Trues), then
compute them together so the files are read only once.

In [ ]:
# Four counts, planned separately, computed together.
n_wage_zero_or_less = (ddf["gross_wage_tzs"] <= 0).sum()
n_wage_too_big = (ddf["gross_wage_tzs"] > MAX_WAGE).sum()
n_no_employer = ddf["employer_no"].isnull().sum()
n_no_posted_date = ddf["posted_date"].isnull().sum()

results = dask.compute(n_wage_zero_or_less, n_wage_too_big, n_no_employer, n_no_posted_date)

print("wage <= 0:          ", results[0])
print("wage above max:     ", results[1])
print("missing employer:   ", results[2])
print("missing posted date:", results[3])

**(3)** Tracing a mismatch back to the raw rows.

In [ ]:
# The worst mismatch.
worst = check.sort_values("diff_share", ascending=False).iloc[0]
print("employer", worst["employer_no"], "period", worst["period"])
print("employer paid:", worst["amount_paid_tzs"], "| member postings add up to:", worst["contributions_tzs"])

In [ ]:
# Its raw member-level rows. This filter is small, so .compute() returns a pandas table.
same_employer = ddf["employer_no"] == worst["employer_no"]
same_period = ddf["period"] == worst["period"]
rows = ddf[same_employer & same_period].compute()

# Highest wages first: an impossible wage would be at the top.
rows.sort_values("gross_wage_tzs", ascending=False).head(8)

Look for a row with an impossible wage, a missing employer number or a missing posted date, or two identical
rows. Removing an impossible wage is right for *features*, but the contribution was probably real. The money
arrived. The fix belongs in the source system (correct the wage), not in silently dropping the row forever.
Write it into the data-quality log, just as you did on Day 1.